# Sprint 9 — IL Emit Engine & Adaptive Promotion

Spec: [`sprint-9-tasks.md`](../../litemapper/docs/requirements/sprint-9-tasks.md) — 13 tasks (S9-T00..T12) lighting up the IL Emit hot path alongside the Sprint 4 Expression Compiler and an adaptive, lock-free Compiled→Emit promotion pipeline.

| Scope                                                  | Task        |
| ------------------------------------------------------ | ----------- |
| `Engine.ILEmit` scaffolding + `CanEmit` capability probe| S9-T00..T01 |
| Flat / null-safe / nested / transformer emit           | S9-T02..T04 |
| `StrategyMode` ladder + selector                       | S9-T05      |
| Per-pair `InvocationCounter`                           | S9-T06      |
| `AdaptivePromotionManager` + bounded queue             | S9-T07      |
| Atomic CAS `DelegateSlot` swap                         | S9-T08      |
| Background `PromotionWorker`                           | S9-T09      |
| `SculptorMeter` + `MappingInspection` extensions       | S9-T10      |
| Parity matrix + concurrent stress                      | S9-T11      |
| CHANGELOG + retro                                      | S9-T12      |

Default `Strategy.Mode = CompiledOnly` so the entire Sprint 8 RC test suite (1241 green) keeps passing unchanged.

## Setup

Reference the Release-built `SmartMapp.Net.dll` (rebuild the solution in Release before running this notebook). The `SculptorMeter` telemetry section below pulls in `System.Diagnostics.DiagnosticSource` for the in-notebook `MeterListener`.

In [ ]:
#r "nuget: System.Diagnostics.DiagnosticSource, 9.0.0"
#r "../src/SmartMapp.Net/bin/Release/net10.0/SmartMapp.Net.dll"

using System.Diagnostics.Metrics;
using System.Collections.Concurrent;
using SmartMapp.Net;
using SmartMapp.Net.Abstractions;
using SmartMapp.Net.Configuration;
using SmartMapp.Net.Engine.Promotion;

Console.WriteLine("SmartMapp.Net loaded — strategy ladder available.");

### Shared fixtures

A pair of flat POCOs the IL Emit pipeline accepts unconditionally (parameterless ctor, `PropertyAccessProvider`-only links). Re-used by every section so the parity assertions can compare apples-to-apples.

In [ ]:
public sealed class Order
{
    public int Id { get; set; }
    public string Buyer { get; set; } = "";
    public int Quantity { get; set; }
    public decimal UnitPrice { get; set; }
}

public sealed class OrderDto
{
    public int Id { get; set; }
    public string Buyer { get; set; } = "";
    public int Quantity { get; set; }
    public decimal UnitPrice { get; set; }
}

public sealed class FlatBlueprint : MappingBlueprint
{
    public override void Design(IBlueprintBuilder plan) => plan.Bind<Order, OrderDto>();
}

static Order MakeOrder(int seed) => new()
{
    Id = seed,
    Buyer = $"Buyer-{seed}",
    Quantity = seed * 3,
    UnitPrice = seed * 9.99m,
};

static ISculptor Forge(StrategyMode mode, int promotionThreshold = 10) =>
    new SculptorBuilder()
        .Configure(opt =>
        {
            opt.Strategy.Mode = mode;
            opt.Strategy.PromotionThreshold = promotionThreshold;
            opt.UseBlueprint<FlatBlueprint>();
        })
        .Forge();

Console.WriteLine("Fixtures ready.");

## 1. `StrategyMode.CompiledOnly` — default, Sprint 8 RC behaviour

The default mode lowers every blueprint through the Sprint 4 Expression Compiler. `MappingInspection.ActiveStrategy` reports `ExpressionCompiled`.

In [ ]:
var sculptor = Forge(StrategyMode.CompiledOnly);
var dto = sculptor.Map<Order, OrderDto>(MakeOrder(1));
var inspect = sculptor.Inspect<Order, OrderDto>();

Console.WriteLine($"output     : Id={dto.Id}, Buyer={dto.Buyer}, Qty={dto.Quantity}, Price={dto.UnitPrice}");
Console.WriteLine($"declared   : {inspect.Strategy}");
Console.WriteLine($"active     : {inspect.ActiveStrategy}");
Console.WriteLine($"promotion  : {inspect.PromotionState?.ToString() ?? "<none>"}");

## 2. `StrategyMode.EmitFirst` — IL Emit on first compile

`EmitFirst` runs every emit-eligible blueprint through `ILEmitMappingCompiler` at Forge time. Non-eligible blueprints (value providers, abstract targets, hooks, etc.) silently fall back to the Expression Compiler.

The DTO output is **byte-identical** to `CompiledOnly` — IL Emit is a back-end choice, not a behaviour change.

In [ ]:
var emitSculptor = Forge(StrategyMode.EmitFirst);
var compiledSculptor = Forge(StrategyMode.CompiledOnly);

var src = MakeOrder(42);
var emittedDto = emitSculptor.Map<Order, OrderDto>(src);
var compiledDto = compiledSculptor.Map<Order, OrderDto>(src);

var parity =
    emittedDto.Id == compiledDto.Id &&
    emittedDto.Buyer == compiledDto.Buyer &&
    emittedDto.Quantity == compiledDto.Quantity &&
    emittedDto.UnitPrice == compiledDto.UnitPrice;

var emitInspect = emitSculptor.Inspect<Order, OrderDto>();
Console.WriteLine($"active     : {emitInspect.ActiveStrategy}");
Console.WriteLine($"parity vs CompiledOnly: {parity}");
Console.WriteLine($"output     : Id={emittedDto.Id}, Buyer={emittedDto.Buyer}, Qty={emittedDto.Quantity}, Price={emittedDto.UnitPrice}");

## 3. `StrategyMode.EmitOnly` — fail fast on non-emittable blueprints

`EmitOnly` is the diagnostic mode for benchmark / proof-of-perf runs. If a blueprint can't lower to IL (e.g. it uses a value provider lambda), `Forge` throws `BlueprintNotEmittableException` with the precise `UnsupportedReason`.

In [ ]:
public sealed class OrderWithTax
{
    public int Id { get; set; }
    public string Buyer { get; set; } = "";
    public decimal UnitPrice { get; set; }
    public decimal Tax { get; set; }   // value provider → not emittable
}

public sealed class NonEmittableBlueprint : MappingBlueprint
{
    public override void Design(IBlueprintBuilder plan) =>
        plan.Bind<Order, OrderWithTax>()
            .Property(d => d.Tax, p => p.From(o => o.UnitPrice * 0.10m));
}

try
{
    _ = new SculptorBuilder()
        .Configure(opt =>
        {
            opt.Strategy.Mode = StrategyMode.EmitOnly;
            opt.UseBlueprint<NonEmittableBlueprint>();
        })
        .Forge();
    Console.WriteLine("(unexpected) Forge succeeded");
}
catch (Exception ex)
{
    Console.WriteLine($"{ex.GetType().Name}");
    Console.WriteLine($"  {ex.Message}");
}

## 4. `StrategyMode.Adaptive` — promotion lifecycle observed

`Adaptive` starts with Expression-Compiled (cheap to compile, AOT-friendly) and lets a background `PromotionWorker` upgrade hot pairs to IL Emit once invocation count crosses `PromotionThreshold`. We set `PromotionThreshold = 3` here so the demo finishes promptly.

Lifecycle: `Cold → Hot → Compiling → Promoted`.

In [ ]:
var sculptor = Forge(StrategyMode.Adaptive, promotionThreshold: 3);

// Pre-promotion view.
sculptor.Map<Order, OrderDto>(MakeOrder(1));
var pre = sculptor.Inspect<Order, OrderDto>();
Console.WriteLine($"before threshold → active={pre.ActiveStrategy}, state={pre.PromotionState}, invocations={pre.InvocationCount}");

// Cross the threshold.
for (var i = 0; i < 5; i++) sculptor.Map<Order, OrderDto>(MakeOrder(i + 2));

// Wait for the background worker to drain.
var deadline = DateTimeOffset.UtcNow.AddSeconds(5);
PromotionState? state = null;
while (DateTimeOffset.UtcNow < deadline)
{
    state = sculptor.Inspect<Order, OrderDto>().PromotionState;
    if (state == PromotionState.Promoted) break;
    await Task.Delay(50);
}

var post = sculptor.Inspect<Order, OrderDto>();
Console.WriteLine($"after  threshold → active={post.ActiveStrategy}, state={post.PromotionState}, invocations={post.InvocationCount}");
Console.WriteLine($"promoted at      → {post.LastPromotedAt:O}");
Console.WriteLine($"compile duration → {post.PromotionCompileDurationMs:0.000} ms");

// Functional parity post-swap.
var dto = sculptor.Map<Order, OrderDto>(MakeOrder(99));
Console.WriteLine($"post-swap output → Id={dto.Id}, Buyer={dto.Buyer}, Qty={dto.Quantity}, Price={dto.UnitPrice}");

## 5. Concurrent mapping during promotion

Under `Adaptive` mode the delegate slot is replaced mid-burst by the promotion worker via `Interlocked.CompareExchange<Func<>>`. Concurrent readers observe either the old (Expression-Compiled) or new (IL Emit) delegate — never a torn reference. Eight threads × 1 000 calls = 8 000 maps, every output asserted byte-correct.

In [ ]:
var sculptor = Forge(StrategyMode.Adaptive, promotionThreshold: 5);

const int threadCount = 8;
const int callsPerThread = 1_000;
var bag = new ConcurrentBag<(int seed, int id, string buyer)>();
var sw = System.Diagnostics.Stopwatch.StartNew();

Parallel.For(0, threadCount, t =>
{
    for (var i = 0; i < callsPerThread; i++)
    {
        var seed = t * callsPerThread + i;
        var output = sculptor.Map<Order, OrderDto>(MakeOrder(seed));
        bag.Add((seed, output.Id, output.Buyer));
    }
});

sw.Stop();
var mismatches = bag.Count(r => r.id != r.seed || r.buyer != $"Buyer-{r.seed}");
var inspect = sculptor.Inspect<Order, OrderDto>();

Console.WriteLine($"total maps          : {bag.Count:N0}");
Console.WriteLine($"mismatches          : {mismatches}");
Console.WriteLine($"elapsed             : {sw.Elapsed.TotalMilliseconds:0.0} ms");
Console.WriteLine($"final active strategy: {inspect.ActiveStrategy}");
Console.WriteLine($"final promotion state: {inspect.PromotionState}");

## 6. Telemetry — `SculptorMeter` + `MeterListener`

Sprint 9 emits three OpenTelemetry-compatible instruments under the `SmartMapp.Net` meter:

| Instrument                                | Type        | Tags                                |
| ----------------------------------------- | ----------- | ----------------------------------- |
| `smartmappnet.cache.promotions`           | `Counter`   | `origin_type`, `target_type`        |
| `smartmappnet.cache.promotion_failures`   | `Counter`   | `origin_type`, `target_type`, `reason` |
| `smartmappnet.cache.compile_duration_ms`  | `Histogram` | `origin_type`, `target_type`        |

All sites are zero-allocation when no listener is registered (the BCL short-circuits inactive counters).

In [ ]:
var events = new ConcurrentBag<string>();
using var listener = new MeterListener();
listener.InstrumentPublished = (instr, l) =>
{
    if (instr.Meter.Name == "SmartMapp.Net") l.EnableMeasurementEvents(instr);
};
listener.SetMeasurementEventCallback<long>((instr, value, tags, _) =>
    events.Add($"{instr.Name,-44} value={value}  tags=[{string.Join(", ", tags.ToArray().Select(t => $"{t.Key}={t.Value}"))}]"));
listener.SetMeasurementEventCallback<double>((instr, value, tags, _) =>
    events.Add($"{instr.Name,-44} value={value:0.000}  tags=[{string.Join(", ", tags.ToArray().Select(t => $"{t.Key}={t.Value}"))}]"));
listener.Start();

var sculptor = Forge(StrategyMode.Adaptive, promotionThreshold: 2);
for (var i = 0; i < 4; i++) sculptor.Map<Order, OrderDto>(MakeOrder(i));

// Drain.
var deadline = DateTimeOffset.UtcNow.AddSeconds(5);
while (DateTimeOffset.UtcNow < deadline &&
       sculptor.Inspect<Order, OrderDto>().PromotionState != PromotionState.Promoted)
{
    await Task.Delay(50);
}

foreach (var evt in events) Console.WriteLine(evt);
if (events.IsEmpty) Console.WriteLine("(no events — promotion may not have fired in time; re-run cell)");

## 7. `MappingInspection` Sprint 9 fields — quick reference

Six new init-only fields surface the IL Emit + promotion lifecycle through the existing inspection API. Under `Adaptive` mode `Sculptor.Inspect<,>` bypasses the inspection cache so each call sees fresh runtime state.

In [ ]:
static void Dump(string label, MappingInspection ins)
{
    Console.WriteLine($"--- {label} ---");
    Console.WriteLine($"  Strategy                   : {ins.Strategy}");
    Console.WriteLine($"  ActiveStrategy             : {ins.ActiveStrategy?.ToString() ?? "<null>"}");
    Console.WriteLine($"  PromotionState             : {ins.PromotionState?.ToString() ?? "<null>"}");
    Console.WriteLine($"  InvocationCount            : {ins.InvocationCount}");
    Console.WriteLine($"  LastPromotedAt             : {ins.LastPromotedAt?.ToString("O") ?? "<null>"}");
    Console.WriteLine($"  PromotionCompileDurationMs : {ins.PromotionCompileDurationMs?.ToString("0.000") ?? "<null>"}");
    Console.WriteLine($"  PromotionError             : {ins.PromotionError?.GetType().Name ?? "<null>"}");
}

foreach (var mode in new[] { StrategyMode.CompiledOnly, StrategyMode.EmitFirst, StrategyMode.Adaptive, StrategyMode.EmitOnly })
{
    var s = Forge(mode, promotionThreshold: 1);
    s.Map<Order, OrderDto>(MakeOrder(1));
    if (mode == StrategyMode.Adaptive)
    {
        // Let the worker run.
        s.Map<Order, OrderDto>(MakeOrder(2));
        var deadline = DateTimeOffset.UtcNow.AddSeconds(3);
        while (DateTimeOffset.UtcNow < deadline &&
               s.Inspect<Order, OrderDto>().PromotionState != PromotionState.Promoted)
        {
            await Task.Delay(50);
        }
    }
    Dump(mode.ToString(), s.Inspect<Order, OrderDto>());
}

## Next

- **`99-acceptance-tests.ipynb`** — assertion-driven regression notebook (Sprint 1–9 coverage).
- **§9.1 BenchmarkDotNet sign-off** — flat ≤ 100 ns, nested ≤ 500 ns, 1 K collection ≤ 100 µs. See `docs/retrospectives/sprint-9-retro.md` *Deferred* section.
- **`v1.0.0` GA tag + NuGet publish** — gated on §9.1 sign-off.